# Hex Maze Session Inventory 

In [ ]:
import numpy as np
import pandas as pd
import datajoint as dj

import spyglass.common as sgc

# Hex maze behavior tables
from spyglass_hexmaze.hex_maze_behavior import (
    HexMazeBlock,
    HexCentroids,
    HexPositionSelection,
    HexPosition,
)
# Hex maze decode tables
from spyglass_hexmaze.hex_maze_decoding import (
    HexMazeDecodedPosition,
    HexMazeDecodedPositionHex,
    HexMazeDecodedHexPath,
)

## Find all sessions in `HexMazeBlock`

In [38]:
# Get all sessions in HexMazeBlock
hex_maze_sessions = sorted(set(HexMazeBlock.fetch('nwb_file_name')))
session_keys = [{'nwb_file_name': s} for s in hex_maze_sessions]

print(f'Found {len(hex_maze_sessions)} hex maze sessions:')
for s in hex_maze_sessions:
    print('   ', s)
    
# Get lab + subject_id for each hex maze session
session_lab = pd.DataFrame((sgc.Session & session_keys).fetch('nwb_file_name', 'subject_id', 'lab_name',
                                                           as_dict=True))

# Get task type
blocks = pd.DataFrame(HexMazeBlock.fetch('nwb_file_name', 'epoch', 'task_type', as_dict=True))
epoch_task = (
    blocks.groupby(['nwb_file_name', 'epoch'])['task_type']
    .agg(lambda x: ', '.join(sorted(set(x))))  # single task type per epoch
    .reset_index()
)

# Attach lab + subject to every epoch
epoch_task = epoch_task.merge(session_lab, on='nwb_file_name', how='left')

# Flag each epoch as barrier vs probability change
epoch_task['is_barrier'] = epoch_task['task_type'].str.contains('barrier', case=False)
epoch_task['is_prob'] = epoch_task['task_type'].str.contains('prob', case=False)

n_sessions = epoch_task['nwb_file_name'].nunique()
n_epochs = len(epoch_task)
print(f'Total hex maze sessions: {n_sessions}')
print(f'Total hex maze epochs:   {n_epochs}')
print(f'Total unique subjects:   {epoch_task["subject_id"].nunique()}')

# Breakdown by lab (sessions, epochs, subjects, task type)
lab_summary = epoch_task.groupby('lab_name').agg(
    n_subjects=('subject_id', 'nunique'),
    n_sessions=('nwb_file_name', 'nunique'),
    n_epochs=('epoch', 'size'),
    n_barrier=('is_barrier', 'sum'),
    n_prob=('is_prob', 'sum'),
)
print('\n By lab: ')
print(lab_summary.to_string())

# Breakdown by subject
subj_summary = epoch_task.groupby('subject_id').agg(
    n_sessions=('nwb_file_name', 'nunique'),
    n_epochs=('epoch', 'size'),
    n_barrier=('is_barrier', 'sum'),
    n_prob=('is_prob', 'sum'),
)
print('\nBy subject')
print(subj_summary.to_string())

# Breakdown by task type
print('\nEpochs by task type')
print(epoch_task['task_type'].value_counts().to_string())

Found 82 hex maze sessions:
    BraveLu20240516_.nwb
    BraveLu20240518_.nwb
    BraveLu20240519_.nwb
    BraveLu20240615_.nwb
    BraveLu20240617_.nwb
    BraveLu20240619_.nwb
    BraveLu20240622_.nwb
    IM-1478_20220719_.nwb
    IM-1478_20220720_.nwb
    IM-1478_20220724_.nwb
    IM-1478_20220725_.nwb
    IM-1478_20220726_.nwb
    IM-1478_20220727_.nwb
    IM-1594_20230725_.nwb
    IM-1594_20230726_.nwb
    IM-1594_20230727_.nwb
    IM-1594_20230728_.nwb
    IM-1830_pacquiao_20250217_.nwb
    IM-1830_pacquiao_20250224_.nwb
    IM-1830_pacquiao_20250226_.nwb
    IM-1830_pacquiao_20250227_.nwb
    IM-1830_pacquiao_20250228_.nwb
    IM-1830_pacquiao_20250407_.nwb
    IM-1830_pacquiao_20250408_.nwb
    IM-1830_pacquiao_20250411_.nwb
    IM-1830_pacquiao_20250414_.nwb
    IM-1830_pacquiao_20250417_.nwb
    IM-1844_elsa_20250408_.nwb
    IM-1844_elsa_20250414_.nwb
    IM-1844_elsa_20250418_.nwb
    IM-1844_elsa_20250423_.nwb
    IM-1844_elsa_20250424_.nwb
    IM-1844_elsa_20250425_.nwb
 

## What data exists for each epoch?

For every hex maze epoch, check whether it has:
- **position** - anything in `PositionOutput`
- **ephys** - anything in `Raw`
- **sorted** - anything in `SpikeSortingOutput`
- **decoded** - anything in `DecodingOutput`
- **theta** - anything in `HexMazeThetaV1`
- **hex decode** - anything in `HexMazeDecodedPosition`

Position, theta and hex decode each resolve to a specific epoch. Ephys, sorting and
decoding are only keyed by session (`Raw` is keyed by `Session`, and the sorting /
decoding merge tables don't carry an epoch), so for those a session's flag applies to
all of its epochs.

In [22]:
import re

import spyglass.spikesorting.v1 as sgs
from spyglass.common import Raw
from spyglass.position import PositionOutput
from spyglass.decoding.decoding_merge import DecodingOutput
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
# Berke lab IM-* sessions have sort groups in spikesorting v1,
# Frank lab sessions have them in spikesorting v0
from spyglass.spikesorting.v0.spikesorting_recording import SortGroup as SortGroupV0
from spyglass_hexmaze.hex_maze_decoding import HexMazeThetaV1

# Only care about hex maze sessions -- the merge tables hold the whole database
hex_session_set = set(hex_maze_sessions)


def merge_part_rows(merge_table, attrs=('nwb_file_name', 'interval_list_name', 'epoch')):
    """Collect the requested attrs from every part of a spyglass merge table.

    Parts of the same merge table key on different attributes! For PositionOutput,
    Trodes/Common position key on interval_list_name, but DLC position keys on epoch,
    and some parts (e.g. PoseV2) don't carry nwb_file_name at all.

    merge_fetch() skips an ENTIRE part if any one requested attribute is missing, which
    silently drops data (e.g. all DLC-tracked sessions). So instead we walk the parts
    ourselves and ask each one only for the attributes it actually has.
    """
    frames = []
    for part in merge_table.parts(as_objects=True):
        have = [a for a in attrs if a in part.heading.names]
        if 'nwb_file_name' not in have:
            continue  # no way to tie this part back to a session, skip it
        df = pd.DataFrame(part.fetch(*have, as_dict=True))
        if len(df):
            df['part'] = part.table_name
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=list(attrs) + ['part'])
    df = pd.concat(frames, ignore_index=True)
    # Make sure every requested column exists, even if no part had it
    for col in attrs:
        if col not in df.columns:
            df[col] = pd.NA
    return df


# Map each session's run interval name ("00_r1", "01_r1", ...) back to its epoch number
task_epochs = pd.DataFrame(
    (sgc.TaskEpoch & session_keys).fetch('nwb_file_name', 'epoch', 'interval_list_name', as_dict=True)
)
interval_to_epoch = {
    (row['nwb_file_name'], row['interval_list_name']): row['epoch']
    for _, row in task_epochs.iterrows()
}

# Run intervals for each session, longest name first, for the prefix match below
run_intervals_by_session = {}
for (nwb, interval), epoch in interval_to_epoch.items():
    run_intervals_by_session.setdefault(nwb, []).append((interval, epoch))
for nwb in run_intervals_by_session:
    run_intervals_by_session[nwb].sort(key=lambda pair: len(pair[0]), reverse=True)


def interval_epoch(nwb_file_name, interval_list_name):
    """Get the epoch an interval belongs to, or None if we can't tell.

    Handles the four naming conventions we see:
      - a run interval straight out of TaskEpoch, e.g. "00_r1" (Berke) or "07_r4" (Frank)
      - a *derived* run interval, e.g. "01_r1_noPreTrialTimes" -- the run interval with
        pre-trial times removed. Frank lab decodes and some theta entries use these.
        They still belong to the epoch of the run interval they're built from.
      - a position interval, e.g. "pos 1 valid times"
      - a hex maze interval built for one epoch, which names the epoch directly:
        "epoch3_block2", "epoch3_barrierShiftInterval1", "epoch7_nonLocal_ALL".
        (Same convention HexMazeBlock.load_from_nwb uses: f"epoch{epoch}_block{block}".)
    """
    name = str(interval_list_name)

    # Exact run interval
    if (nwb_file_name, name) in interval_to_epoch:
        return interval_to_epoch[(nwb_file_name, name)]

    # Derived run interval -- match the run interval it starts with.
    # Sorted longest-first so e.g. "01_r10" can't be swallowed by "01_r1".
    for interval, epoch in run_intervals_by_session.get(nwb_file_name, []):
        if name.startswith(interval):
            return epoch

    # Position interval
    match = re.match(r'pos (\d+) valid times', name)
    if match:
        return int(match.group(1))

    # Hex maze interval that names its epoch up front (block / barrier shift / nonlocal)
    match = re.match(r'epoch(\d+)_', name)
    if match:
        return int(match.group(1))

    return None


def resolve_position_epoch(row):
    """Get the epoch for a position entry.

    DLC parts store `epoch` directly. Trodes/Common parts instead store an
    interval_list_name like "pos 1 valid times", so we parse the epoch back out.
    """
    if pd.notna(row['epoch']):
        return int(row['epoch'])
    return interval_epoch(row['nwb_file_name'], row['interval_list_name'])


def report_unmapped(df, interval_col, label):
    """Print any entries whose interval didn't resolve to an epoch, instead of
    silently dropping them (which would under-report the data type)."""
    unmapped = df[df['resolved_epoch'].isna()]
    if len(unmapped):
        print(f'!! {len(unmapped)} {label} entry(s) whose {interval_col} did not map to an epoch:')
        print(unmapped[interval_col].value_counts().to_string(), '\n')


def sort_group_table(nwb_file_name):
    """Return the SortGroup table (v1 or v0) that holds this session's sort groups."""
    return sgs.SortGroup if nwb_file_name.startswith('IM-') else SortGroupV0


# ---------- Raw ephys ----------
# Raw is keyed by Session, so if a session has raw ephys then all of its epochs do
ephys_sessions = set((Raw & session_keys).fetch('nwb_file_name'))

# ---------- Sort groups ----------
# Having raw ephys is NOT the same as being ready for LFP/theta! The theta pipeline pulls
# its electrodes from the SortGroup table, so a session with raw ephys but no sort groups
# gets skipped entirely. This column is what makes has_theta <= has_sort_group <= has_ephys
# readable (e.g. BraveLu has 27 epochs of raw ephys but only 7 with sort groups).
sort_group_sessions = {
    nwb_file_name for nwb_file_name in hex_maze_sessions
    if len(sort_group_table(nwb_file_name) & {'nwb_file_name': nwb_file_name}) > 0
}

# ---------- Spike sorting (SpikeSortingOutput) ----------
# Sorting is always done per session, so this is a session-level flag by design.
# NOTE: do NOT use merge_fetch('nwb_file_name') here! The v1 part (CurationV1) is keyed by
# sorting_id, not nwb_file_name, so it would get skipped and we'd miss every v1-sorted
# session. get_restricted_merge_ids() walks back to SpikeSortingRecordingSelection for v1
# and CuratedSpikeSorting for v0, so it catches both pipelines.
sorted_sessions = set()
for nwb_file_name in hex_maze_sessions:
    merge_ids = SpikeSortingOutput().get_restricted_merge_ids(
        {'nwb_file_name': nwb_file_name}, sources=['v0', 'v1'], restrict_by_artifact=False
    )
    if len(merge_ids):
        sorted_sessions.add(nwb_file_name)

# ---------- Position (PositionOutput) ----------
pos_df = merge_part_rows(PositionOutput())
pos_df = pos_df[pos_df['nwb_file_name'].isin(hex_session_set)].copy()

position_epochs = set()
if len(pos_df):
    pos_df['resolved_epoch'] = pos_df.apply(resolve_position_epoch, axis=1)
    report_unmapped(pos_df, 'interval_list_name', 'position')
    with_epoch = pos_df.dropna(subset=['resolved_epoch'])
    position_epochs = set(zip(with_epoch['nwb_file_name'], with_epoch['resolved_epoch'].astype(int)))

# ---------- Decoding (DecodingOutput) ----------
# The decoding selection tables key on IntervalList.proj(decoding_interval=...), so every
# decode entry records the interval it was run on -- which gives us the epoch.
# Berke lab decodes use the bare run interval ("00_r1"); Frank lab decodes use the derived
# interval ("01_r1_noPreTrialTimes"). interval_epoch() handles both.
dec_df = merge_part_rows(DecodingOutput(), attrs=('nwb_file_name', 'decoding_interval'))
dec_df = dec_df[dec_df['nwb_file_name'].isin(hex_session_set)].copy()

decode_epochs = set()
if len(dec_df):
    dec_df['resolved_epoch'] = [
        interval_epoch(nwb, interval)
        for nwb, interval in zip(dec_df['nwb_file_name'], dec_df['decoding_interval'])
    ]
    report_unmapped(dec_df, 'decoding_interval', 'decode')
    with_epoch = dec_df.dropna(subset=['resolved_epoch'])
    decode_epochs = set(zip(with_epoch['nwb_file_name'], with_epoch['resolved_epoch'].astype(int)))

# ---------- Theta (HexMazeThetaV1) ----------
# HexMazeThetaV1 inherits from LFPBandV1, so it carries nwb_file_name and
# target_interval_list_name (the run interval, e.g. "00_r1") -> map that back to an epoch
theta_df = pd.DataFrame(
    (HexMazeThetaV1 & session_keys).fetch(
        'nwb_file_name', 'target_interval_list_name', as_dict=True
    )
)
theta_epochs = set()
if len(theta_df):
    theta_df['resolved_epoch'] = [
        interval_epoch(nwb, interval)
        for nwb, interval in zip(theta_df['nwb_file_name'], theta_df['target_interval_list_name'])
    ]
    report_unmapped(theta_df, 'target_interval_list_name', 'theta')
    with_epoch = theta_df.dropna(subset=['resolved_epoch'])
    theta_epochs = set(zip(with_epoch['nwb_file_name'], with_epoch['resolved_epoch'].astype(int)))

# ---------- Hex maze decode tables (HexMazeDecodedPosition) ----------
# Keyed by DecodingOutput merge id + TaskEpoch, so nwb_file_name + epoch come for free
hex_dec_df = pd.DataFrame(
    (HexMazeDecodedPosition & session_keys).fetch('nwb_file_name', 'epoch', as_dict=True)
)
hex_decode_epochs = set()
if len(hex_dec_df):
    hex_decode_epochs = set(zip(hex_dec_df['nwb_file_name'], hex_dec_df['epoch']))


def epoch_flag(epoch_set):
    """Flag each row of epoch_task by whether its (session, epoch) is in epoch_set."""
    return [
        (nwb, ep) in epoch_set
        for nwb, ep in zip(epoch_task['nwb_file_name'], epoch_task['epoch'])
    ]


# ---------- Add the flags to our epoch table ----------
# position, decode, theta and hex decode all resolve to a specific epoch.
# ephys, sort groups and sorting are session-level (Raw and SortGroup are keyed by session,
# and sorting is always done per session), so a session's flag applies to all of its epochs.
epoch_task['has_ephys'] = epoch_task['nwb_file_name'].isin(ephys_sessions)
epoch_task['has_sort_group'] = epoch_task['nwb_file_name'].isin(sort_group_sessions)
epoch_task['has_position'] = epoch_flag(position_epochs)
epoch_task['has_sorting'] = epoch_task['nwb_file_name'].isin(sorted_sessions)
epoch_task['has_decode'] = epoch_flag(decode_epochs)
epoch_task['has_theta'] = epoch_flag(theta_epochs)
epoch_task['has_hex_decode'] = epoch_flag(hex_decode_epochs)

data_cols = ['has_position', 'has_ephys', 'has_sort_group', 'has_theta',
             'has_sorting', 'has_decode', 'has_hex_decode']

# ---------- Report everything by epoch ----------
n_epochs = len(epoch_task)
epoch_counts = epoch_task[data_cols].sum().astype(int)

print(f'Epochs with each data type (out of {n_epochs} epochs):')
for col in data_cols:
    print(f'  {col:16} {epoch_counts[col]:4} / {n_epochs}')

# By lab, with total epochs as the first column
lab_data = epoch_task.groupby('lab_name')[data_cols].sum().astype(int)
lab_data.insert(0, 'n_epochs', epoch_task.groupby('lab_name').size())
print('\nBy lab (epochs):')
print(lab_data.to_string())

# By subject, with total epochs as the first column
subj_data = epoch_task.groupby('subject_id')[data_cols].sum().astype(int)
subj_data.insert(0, 'n_epochs', epoch_task.groupby('subject_id').size())
print('\nBy subject (epochs):')
print(subj_data.to_string())

Epochs with each data type (out of 178 epochs):
  has_position      146 / 178
  has_ephys         154 / 178
  has_sort_group     83 / 178
  has_theta          83 / 178
  has_sorting        52 / 178
  has_decode         47 / 178
  has_hex_decode     39 / 178

By lab (epochs):
             n_epochs  has_position  has_ephys  has_sort_group  has_theta  has_sorting  has_decode  has_hex_decode
lab_name                                                                                                          
Berke Lab          50            30         26              24         24           22          22              19
Loren Frank       128           116        128              59         59           30          25              20

By subject (epochs):
                  n_epochs  has_position  has_ephys  has_sort_group  has_theta  has_sorting  has_decode  has_hex_decode
subject_id                                                                                                             
Br

## Full epoch inventory

One row per hex maze epoch: lab, subject, session, epoch, and what data exists for it.

In [23]:
# One row per epoch: lab, subject, session, epoch, then the has_* data columns
epoch_inventory = epoch_task[
    ['lab_name', 'subject_id', 'nwb_file_name', 'epoch'] + data_cols
].copy()

epoch_inventory = epoch_inventory.rename(
    columns={'lab_name': 'lab', 'subject_id': 'subject', 'nwb_file_name': 'session'}
)

epoch_inventory = epoch_inventory.sort_values(
    ['lab', 'subject', 'session', 'epoch']
).reset_index(drop=True)

print(f'{len(epoch_inventory)} epochs')
display(epoch_inventory)

178 epochs


,lab,subject,session,epoch,has_position,has_ephys,has_sort_group,has_theta,has_sorting,has_decode,has_hex_decode
0,Berke Lab,IM-1478,IM-1478_20220719_.nwb,0,True,True,True,True,True,True,True
1,Berke Lab,IM-1478,IM-1478_20220720_.nwb,0,True,True,True,True,True,True,True
2,Berke Lab,IM-1478,IM-1478_20220724_.nwb,0,True,True,True,True,True,True,True
3,Berke Lab,IM-1478,IM-1478_20220725_.nwb,0,True,True,True,True,True,True,True
4,Berke Lab,IM-1478,IM-1478_20220726_.nwb,0,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...
173,Loren Frank,Toby,Toby20250329_.nwb,1,True,True,False,False,False,False,False
174,Loren Frank,Toby,Toby20250329_.nwb,3,True,True,False,False,False,False,False
175,Loren Frank,Toby,Toby20250329_.nwb,5,True,True,False,False,False,False,False
176,Loren Frank,Toby,Toby20250330_.nwb,1,True,True,False,False,False,False,False
